<a href="https://colab.research.google.com/github/Kimballa05/agentic-ai-jetstream2-bioinformatics/blob/main/agentic_ai_ecoli_wgs_workshop_participant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI for *E. coli* sequencing QC

**Worshop scenario:** A research project requires several whole genome sequencing (WGS) datasets of *Escherichia coli* (*E. coli*), a bacterium commonly studied in microbiology. Before you can use sequencing data to study a bacterial genome, you need to know **what was sequenced, whether the correct files are present, and whether the sequencing reads are reliable**.

You are tasked to investigate these questions and decide to do so with the help of an AI coding agent. The agent can propose commands, process files, and create figures, but **you are responsible for checking whether its results support its conclusions**.

Note: Sample information, sequencing files, and a small demonstration dataset for machine learning has already been prepared for this exercise. You **will not need to download data, and any bioinformatics software needed for this workshop should already be installed**. If a file is missing, ask an instructor rather than downloading a substitute.


First, we'll identify the datasets appropriate for this analysis. Next, we'll check and clean their sequencing reads. We'll then compare results with a few plots, try a small machine-learning demonstration, and prepare a short report of our findings and decision-making.



## Working with your AI agent

We will use **OpenCode** as our AI coding agent in our JupyterLab environment. Because of this, the notebook gives you questions and example prompts, with  few prewritten code cells made to create directories or view tables and other visualizations.


When you see **Ask your agent**, you can paste the suggested text into OpenCode. However, it's not necessary to follow the suggestion exactly to achieve the desired results. When you see **Explore**, you can experiment with follow-up questions or figure generations. When you see **Checkpoint**, be sure to independently confirm a result as satisfactory before moving on.




## Your roadmap



| Stage | Learning Objective | Output to look into |
|---|---|---|
| 0. Find the files | Understand the prepared dataset and directory structure | File verification |
| 1. Review metadata | Decide which samples match our WGS criteria | Accepted/reviewed sample table |
| 2. Check sequencing files | Understand paired-end R1/R2 reads and verify files | FASTQ inventory |
| 3. Explore raw quality | See how sequencing quality is measured | QC report and plot |
| 4. Preprocess reads | Learn what trimming/filtering changes | Cleaned reads and `fastp` report |
| 5. Compare results | Decide whether cleaning helped | Before/after figures |
| 6. Summarize measurements | Turn reports into a table for comparison | QC feature table |
| 7. Use ML classifiers | Explore PASS/REVIEW predictions of processed samples | ML results and plot |
| 8. Report findings | Separate evidence from assumptions | Short final report |


## 0 — Get oriented: what files are we working with?

Before analyzing data, we need to know what we were given. In this workshop, a **sample** is one sequencing dataset represented by a sample label such as `sample01`. A **run accession** is an identifier assigned by a public sequencing archive (an identifier beginning with `SRR` or `ERR` comes from NCBI SRA database, for example). It lets researchers trace a dataset back to its source.

You'll see four main types of input:

- **`samples.tsv`** — a short manifest linking local sample labels (`sample_id`) to public sequencing run identifiers (`run_accession`). A TSV is a table whose columns are separated by tabs.
- **`sra_metadata.tsv`** — information *about* each sequencing experiment, such as the organism and sequencing approach; not the DNA sequence data.
- **`*.fastq.gz`** — compressed files containing the actual sequencing reads, including nucleotide data and base-quality information. Since these are paired-end reads, each sample will have an *_1.fastq.gz (R1) and a *_2.fastq.gz (R2) file. We will explain R1 and R2 in Step 2.
- **`synthetic_qc_training.tsv`** — an artificial dataset used later to illustrate machine learning. It is **not** measured sequencing data from our samples.

Here is the directory outline that will be used for this workshop. The rest of the workshop will be run on the assumption that this is the directory structure.

```text
project/
├── inputs/                         # provided files
│   ├── samples.tsv
│   ├── metadata/sra_metadata.tsv
│   ├── raw/<sample_id>/*_1.fastq.gz, *_2.fastq.gz
│   └── ml/synthetic_qc_training.tsv
├── metadata/                       # your decisions about which samples to analyze
├── qc/raw/                         # measurements on original reads
├── processed/                      # cleaned FASTQ files
├── qc/processed/                   # measurements on cleaned reads
├── reports/                        # tables, plots, and final report
├── scripts/                        # code created with the agent
└── logs/                           # commands and software versions
```

### Ask your agent

> Help me find `samples.tsv`, `sra_metadata.tsv`, paired-end FASTQ files, and synthetic QC training table. First show me the paths and a proposed folder structure for my private outputs. Do not download anything or change the shared files. Create `reports/input_inventory.tsv` listing the files found, sizes, and any missing expected files. Record the commands you run in `logs/commands.log`.

**Explore:** Ask the agent to explain which files contain data, which contain descriptions of data, and which folders are safe to write to.

**Checkpoint:** Open the inventory and confirm that at least two reported file paths truly exist. If the expected inputs aren't there, you can ask an instructor for help on retrieving missing inputs.

In [ ]:
# Optional *read-only* inspection helper
# Enter your actual personal project directory after your instructor provides it.
from pathlib import Path
PROJECT = Path(".").resolve()  # Change this to your own project root when needed.
print("Current directory:", PROJECT)
print("Nearby entries:", [p.name for p in sorted(PROJECT.iterdir())][:20])

## 1 — Understand the samples before analyzing them

Our research question is about **paired-end *E. coli* whole-genome sequencing (WGS)**. WGS aims to obtain sequence information from across an organism's genome. However, not every dataset labeled with *E. coli* was generated for this purpose. For example, **16S sequencing** targets a specific marker gene region to investigate microbial identity or community composition; it is not equivalent to sequencing a complete bacterial genome.

The provided sample list contains several records, some including **an intentionally unsuitable sequencing approach** for this WGS exercise. We will use the metadata to identify it rather than trusting filenames or guessing which dataset to keep.

### Reading the metadata

| Field | Meaning | Importance |
|---|---|---|
| `sample_id` | User-created name (`sample01`) | Connects our results and figures to the right files |
| `run_accession` | Public sequencing run identifier (`SRR...`, `ERR...`, etc.) | Traces the run to the original archive record |
| `organism` | Organism recorded for the dataset | Should be consistent with the *E. coli* objective |
| `strategy` | Type of sequencing experiment, such as `WGS` or `AMPLICON` | Distinguishes genome-wide from targeted sequencing |
| `source` | General origin of sequenced material, such as genomic DNA | Helps interpret the experiment |
| `layout` | Whether reads were generated as `PAIRED` or `SINGLE` | Determines whether the paired-read workflow is appropriate |
| `instrument` | Sequencing machine/model, if provided | Gives context about how data were generated |

**Important:** These are metadata *descriptions*, not guarantees of correctness. Missing or conflicting fields deserve a `REVIEW` decision. We should be able to explain every sample we include or exclude.

### Your task

Decide which provided runs are suitable for the subsequent paired-end WGS analysis, and document your reasoning before processing any FASTQ files.

### Ask your agent

> Read the **provided** `samples.tsv` and `metadata/sra_metadata.tsv`. Before writing code, explain which metadata fields are relevant to deciding whether a sample is paired-end *E. coli* WGS. Make a small table with `sample_id`, `run_accession`, `organism`, `strategy`, `layout`, `decision`, and `reason`. Save it as `metadata/metadata_review.tsv`, and save the selected samples as `metadata/valid_wgs.tsv`. Flag missing or ambiguous metadata for review.

**Explore:** Ask the agent to explain why a dataset can be genuine *E. coli* sequencing data but still be unsuitable for *this* analysis.

**Checkpoint:** Open `metadata_review.tsv`. Can you point to the exact metadata field(s) that justify each decision? The accepted list will guide the next step.

In [ ]:
# Read-only viewer: run after the agent saves the review table.
import pandas as pd
p = PROJECT / "metadata" / "metadata_review.tsv"
if p.exists():
    display(pd.read_csv(p, sep="\t"))
else:
    print("Not ready yet:", p)

## 2 — Check the raw sequencing files: what are R1 and R2?

DNA sequencing machines generally do not deliver one complete bacterial chromosome in a single piece. Instead, they generate many shorter **reads**, or strings of DNA bases (`A`, `C`, `G`, and `T`) measured from pieces of the genome.

For **paired-end sequencing**, the machine reads **both ends of the same DNA fragment**. Those two observations are commonly stored in separate files:

- **R1 (Read 1)** — the sequence read from one end of a fragment, often in a file named `*_1.fastq.gz` or `*_R1.fastq.gz`.
- **R2 (Read 2)** — the matching read from the other end of that fragment, often `*_2.fastq.gz` or `*_R2.fastq.gz`.

When represented correctly, there should usually be one R1 record for every R2 record, with corresponding identifiers. Matching counts are a useful first check, but they do **not** establish that individual read identifiers are correctly paired or that every read is biologically trustworthy.

A **FASTQ** record has 4 parts: a read identifier (heading), the nucleotide sequence, a separator line that starts with a (+) sign, and a per-base quality string. The `.gz` extension means the file is compressed to save storage space.

### Your task

Use the accepted sample list from Step 1 to find each sample's two mate files and check that they're usable before any processing begins.

### Ask your agent

> Using `metadata/valid_wgs.tsv` and the staged FASTQ directory, identify the R1 and R2 files for every accepted sample. Check that each file exists, is nonempty, can be read as gzip-compressed FASTQ, and that the pairs have matching read counts. Make `reports/fastq_inventory.tsv` summarizing the results. Do not alter the original files.

**Explore:** Ask why a file can exist and still be unusable. What additional check would help confirm that the two mates actually belong together?

**Checkpoint:** For each accepted sample, locate its R1 and R2 filenames and review the agent's count comparison. Do not proceed with a missing mate or corrupted file.


## 3 — Investigate the quality of the original reads

Now that we've checked *which* files we have, we can ask whether the sequencing data are of useful quality. During sequencing, a machine estimates which DNA base appears at each position. That estimate may be uncertain, especially near the ends of reads. Artificial sequences called **adapters** may also remain in the files from the sequencing preparation process.

Some basic measurements you may encounter are:

- **Read count:** how many sequence reads were recorded; paired-end data have both R1 and R2 files.
- **Read length:** the number of DNA bases in a read.
- **Phred quality score:** a way of expressing confidence in a called base; higher scores mean lower estimated error probabilities. **Q30** corresponds to an estimated error probability of 1 in 1,000 for a base call.
- **Adapter content:** evidence of leftover laboratory adapter sequences that may need trimming.

We can use **SeqKit** for simple FASTQ statistics, **FastQC** for per-file quality diagnostics, and **MultiQC** to combine multiple QC reports into a single overview. A tool's warning flag is a reason to investigate, not automatically a failed sample.

### Your task

Ask your agent to inspect the *original* reads and produce both a report and a plot that helps you compare samples.

### Ask your agent

> Propose a lightweight QC workflow for the accepted FASTQs using SeqKit, FastQC, and MultiQC. Before running it, show the commands, expected runtime considerations, and output locations. After I approve, save results under `qc/raw/`, produce a short summary of the most important quality metrics, and make **one simple plot** comparing read counts or lengths among samples. Explain the figure in plain language and note any concerns that need confirmation.

**Explore:** Ask the agent to make the figure easier for a beginner to read by adding a title, units, and clear sample labels. Does the figure really support its explanation?

**Checkpoint:** Open an actual FastQC or MultiQC report. Compare at least one metric with the agent's written summary before moving on.


## 4 — Clean the sequencing reads

Raw sequencing data often need **preprocessing** before downstream analysis. For this exercise, we'll use `fastp`, a tool that can identify/remove adapters, trim low-quality regions, and filter reads according to chosen criteria.

Think of trimming as removing unreliable portions of a measurement—not changing the biological identity of a sample. But cleaning is a tradeoff: aggressive filtering may remove more questionable data while also discarding useful information. In paired-end data, both mates should be handled consistently, and the original FASTQs should stay untouched.

`fastp` can produce **new R1 and R2 FASTQ files** alongside **HTML/JSON reports** summarizing what happened. The HTML file is a convenient way to read and visualize results on a webpage, while the JSON file is structured data that can be used for tables and plots.

### Your task

Have the agent propose a conservative, reproducible preprocessing strategy, explain the parameters, and run it only after you approve.

### Ask your agent

> Propose paired-end read preprocessing with `fastp` for the accepted WGS samples. Use conservative, clearly stated settings. Do not overwrite raw input data. First tell me which settings you propose, why, and where the cleaned FASTQs and JSON/HTML reports will go. Only run the commands after my approval. Record tool version, exact commands, and sample-specific outputs in `logs/`. Stop and report errors rather than silently skipping samples.

**Explore:** Ask what might happen if the minimum accepted read length or quality threshold were much stricter. What could improve, and what could be lost?

**Checkpoint:** Find each sample's processed R1/R2 pair and its `fastp` report. Compare the reported number of reads before and after filtering.
